In [1]:
import requests
import pandas as pd
import time

url = "https://www.peeringdb.com/api/fac"


eu27_codes = [
    "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI",
    "FR", "DE", "GR", "HU", "IE", "IT", "LV", "LT", "LU",
    "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE"
]

all_facilities = []
limit = 100
skip = 0

while True:
    params = {
        "country__in": ",".join(eu27_codes),
        "fields": "id,name,country,city,latitude,longitude,org_id,status",
        "limit": limit,
        "skip": skip
    }

    response = requests.get(url, params=params, timeout=30)

    if response.status_code == 429:
        print("Rate limit reached. Waiting 65 seconds...")
        time.sleep(65)
        continue

    response.raise_for_status()

    batch = response.json()["data"]

    if not batch:
        break

    all_facilities.extend(batch)
    print(f"Downloaded {len(batch)} records — total: {len(all_facilities)}")

    if len(batch) < limit:
        break

    skip += limit

    time.sleep(3.5)

facilities_df = pd.DataFrame(all_facilities)

facilities_df = facilities_df.drop_duplicates(subset="id")

facilities_df.to_csv(
    "eu_peeringdb_facilities.csv",
    index=False
)

facilities_df.head()

Downloaded 100 records — total: 100
Downloaded 100 records — total: 200
Downloaded 100 records — total: 300
Downloaded 100 records — total: 400
Downloaded 100 records — total: 500
Downloaded 100 records — total: 600
Downloaded 100 records — total: 700
Downloaded 100 records — total: 800
Downloaded 100 records — total: 900
Downloaded 100 records — total: 1000
Downloaded 100 records — total: 1100
Downloaded 100 records — total: 1200
Downloaded 100 records — total: 1300
Downloaded 100 records — total: 1400
Downloaded 36 records — total: 1436


,id,org_id,name,status,city,country,latitude,longitude,website,available_voltage_services
0,18,31,NIKHEF Amsterdam,ok,Amsterdam,NL,52.356394,4.950837,https://www.nikhef.nl/,NaN
1,52,41935,UltraEdge Velizy,ok,Paris,FR,48.782499,2.208747,http://www.ultraedge.com/,NaN
2,53,7,Telehouse - Paris 2 (Voltaire - Léon Frot),ok,Paris,FR,48.856654,2.385320,http://www.telehouse.com,NaN
3,54,7,Telehouse - Paris 1 (Jeûneurs),ok,Paris,FR,48.869869,2.344236,http://www.telehouse.com,NaN
4,55,41935,UltraEdge Paris - Courbevoie,ok,Paris,FR,48.903746,2.257694,http://www.ultraedge.com/,NaN


In [2]:
facilities_df_cleaned=facilities_df.drop(columns=["website","available_voltage_services" ])

In [3]:
facilities_df_cleaned = facilities_df_cleaned.rename(columns={"country": "geo"})

In [4]:
facilities_df_cleaned = facilities_df_cleaned.drop(
    columns=["id", "org_id", "name", "status", "latitude", "longitude"]
)

In [5]:
facilities_df_cleaned

,city,geo
0,Amsterdam,NL
1,Paris,FR
2,Paris,FR
3,Paris,FR
4,Paris,FR
...,...,...
1431,Hannover,DE
1432,Lisbon,PT
1433,Brasov,RO
1434,LJubljana,SI


In [7]:
facilities_df_cleaned.to_csv(
    "eu_facilities_cleaned.csv")